# AML s26 - Logistic Regression

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


### Load and show data

In [2]:
df = pd.read_csv('titanic.csv')

print(df.head(), "\n")

   survived  pclass     sex   age  sibsp  parch     fare embarked
0         0       3    male  22.0      1      0   7.2500        S
1         1       1  female  38.0      1      0  71.2833        C
2         1       3  female  26.0      0      0   7.9250        S
3         1       1  female  35.0      1      0  53.1000        S
4         0       3    male  35.0      0      0   8.0500        S 



### Preprocess data (split into X and y, deal with missing data, and encode categoricals)

In [3]:
features = ["pclass", "sex", "age", "fare", "embarked"]
target   = "survived"

data = df[features + [target]].copy()
data = data.dropna(subset=[target])  # no class label --> we cannot train or test! 

data["age"]      = data["age"].fillna(data["age"].median())
data["fare"]     = data["fare"].fillna(data["fare"].median())
data["embarked"] = data["embarked"].fillna("S")

# Encode categoricals
data["sex"]      = (data["sex"] == "male").astype(int)   # male=1, female=0
data["embarked"] = data["embarked"].map({"S": 0, "C": 1, "Q": 2})

X = data[features]
y = data[target].astype(int)

print(f"Samples: {len(X)}  |  Survival rate: {y.mean():.1%}\n")

Samples: 891  |  Survival rate: 38.4%



### Train / Test Split (note we use "stratify=y")

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Fit with statsmodels

In [5]:
X_train_sm = sm.add_constant(X_train)
X_test_sm  = sm.add_constant(X_test)

model = sm.Logit(y_train, X_train_sm)
result = model.fit(disp=False)   # 'disp=False' suppresses the iteration log

### Get Full statsmodels Summary

In [6]:
print("=== Statsmodels Logit Summary ===")
print(result.summary())

=== Statsmodels Logit Summary ===
                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  712
Model:                          Logit   Df Residuals:                      706
Method:                           MLE   Df Model:                            5
Date:                Tue, 24 Feb 2026   Pseudo R-squ.:                  0.3376
Time:                        10:13:28   Log-Likelihood:                -313.96
converged:                       True   LL-Null:                       -473.99
Covariance Type:            nonrobust   LLR p-value:                 4.869e-67
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.6876      0.576      8.139      0.000       3.559       5.817
pclass        -1.2137      0.157     -7.747      0.000      -1.521      -0.907
sex           -2.5

### Evaluate on Test Set 

In [7]:
y_pred_prob = result.predict(X_test_sm)
y_pred      = (y_pred_prob >= 0.5).astype(int)

print(f"\n=== Test Set Accuracy ===")
print(f"{accuracy_score(y_test, y_pred):.3f}\n")

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

print("=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm,
      index=["Actual: Died", "Actual: Survived"],
      columns=["Pred: Died", "Pred: Survived"]), "\n")



=== Test Set Accuracy ===
0.771

=== Classification Report ===
              precision    recall  f1-score   support

        Died       0.80      0.84      0.82       110
    Survived       0.72      0.67      0.69        69

    accuracy                           0.77       179
   macro avg       0.76      0.75      0.75       179
weighted avg       0.77      0.77      0.77       179

=== Confusion Matrix ===
                  Pred: Died  Pred: Survived
Actual: Died              92              18
Actual: Survived          23              46 



### Predict New Passenger

In [10]:
# Encoding: sex male=1 female=0 | embarked S=0 C=1 Q=2
passengers = pd.DataFrame([
    [1, 0, 30, 100, 1],   # 1st-class female, age 30, fare £100, Cherbourg
    [3, 1, 25,   8, 0],   # 3rd-class male,   age 25, fare  £8,  Southampton
], columns=features)

passengers_sm = sm.add_constant(passengers, has_constant="add")
probs = result.predict(passengers_sm)

print("=== Survival Predictions for New Passengers ===")
print(f"  1st-class female, age 30, fare £100 => {probs.iloc[0]:.1%} survival probability")
print(f"  3rd-class male,   age 25, fare  £8  => {probs.iloc[1]:.1%} survival probability")

=== Survival Predictions for New Passengers ===
  1st-class female, age 30, fare £100 => 94.6% survival probability
  3rd-class male,   age 25, fare  £8  => 8.1% survival probability
